# 01 — Getting Started with Pyvium

**Pyvium** is a Python wrapper around the IviumSoft "Software Development Driver DLL" that lets you control Ivium electrochemistry devices (CompactStat, etc.) programmatically.

### Requirements

- **Windows only** — the underlying DLL is Windows-specific
- **IviumSoft must be installed** (version 4.1239 or compatible)
- **IviumSoft must be running** before calling most Pyvium functions
- Python 3.11 or 3.12

---

## Library Architecture

Pyvium exposes three independent layers:

```
┌─────────────────────────────────────────────────────────┐
│  from pyvium import Pyvium   ← use this for experiments │
│  High-level wrapper — Pythonic names, raises exceptions │
├─────────────────────────────────────────────────────────┤
│  from pyvium import Core     ← low-level DLL calls      │
│  Raw DLL wrappers — returns (result_code, value) tuples │
├─────────────────────────────────────────────────────────┤
│  from pyvium import Tools    ← no hardware needed       │
│  IDF file parsing, CSV export — works offline           │
└─────────────────────────────────────────────────────────┘
```

This series of notebooks covers the **Pyvium** layer (the recommended API). See notebook `07_data_processing.ipynb` for the Tools layer.

## 1. Importing the Library

In [ ]:
from pyvium import Pyvium

# Import error types for explicit exception handling
from pyvium.errors import (
    DriverNotOpenError,
    IviumSoftNotRunningError,
    DeviceNotConnectedToIviumSoftError,
    NoDeviceDetectedError,
    DeviceBusyError,
)

print("pyvium imported successfully")

## 2. Version Verification

Before connecting to any hardware, check that the DLL bundled with pyvium is compatible with the installed IviumSoft version. A version mismatch will cause silent failures or crashes.

> **Note:** `open_driver()` must be called before any other function. It establishes the communication channel with IviumSoft.

In [ ]:
Pyvium.open_driver()

print(f"DLL version       : {Pyvium.get_dll_version()}")
print(f"DLL version string: {Pyvium.get_dll_version_string()}")
print(f"IviumSoft version : {Pyvium.get_iviumsoft_version()}")
print(f"Required DLL ver  : {Pyvium.get_version_host()}")
print(f"Version match     : {Pyvium.check_dll_version()}")

## 3. Driver Lifecycle

The driver is the communication channel between Python and IviumSoft. Always open it before use and close it when done.

| Step | Method | Effect |
|------|--------|--------|
| 1 | `open_driver()` | Connects Python to the running IviumSoft process |
| 2 | *(do work)* | Measure, configure, run methods... |
| 3 | `close_driver()` | Releases the connection — IviumSoft keeps running |

Calling `open_driver()` when the driver is already open automatically closes and reopens it (safe to call multiple times).

In [ ]:
# The recommended pattern: use try/finally to guarantee the driver is closed
try:
    Pyvium.open_driver()
    print("Driver open — ready to communicate with IviumSoft")

    # ... your measurement code goes here ...

finally:
    Pyvium.close_driver()
    print("Driver closed")

## 4. Error Handling

The Pyvium layer translates raw DLL return codes into typed Python exceptions. Each exception maps to a specific device state or command result:

| Exception | DLL code | Meaning | Fix |
|-----------|----------|---------|-----|
| `DriverNotOpenError` | — | `open_driver()` was not called | Call `open_driver()` first |
| `IviumSoftNotRunningError` | — | IviumSoft process is not running | Start IviumSoft |
| `DeviceNotConnectedToIviumSoftError` | — | Device found but not connected in IviumSoft | Click Connect in IviumSoft, or call `connect_device()` |
| `NoDeviceDetectedError` | `-1` | No device detected (also raised by setter result code -1) | Check USB cable and driver installation |
| `DeviceBusyError` | — | Device is already running a measurement | Wait or call `abort_method()` |
| `CellOffError` | — | Tried to measure with cell off | Call `set_cell_on()` first |
| `IllegalCommandError` | `1` | Command not valid for this device or its current mode | Check connection mode and device capabilities |
| `InvalidStateError` | `3` | Command is valid but device state prevents it | Wait for any running measurement to finish |
| `ValueError` (built-in) | `2` | Argument out of range — firmware rejected the value | Check the valid range for the parameter |

In [ ]:
# Example: handling errors gracefully
try:
    Pyvium.open_driver()
    Pyvium.connect_device()
    print("Device connected")

    # ... measurement code ...

except IviumSoftNotRunningError:
    print("ERROR: IviumSoft is not running. Please start it and try again.")

except DeviceNotConnectedToIviumSoftError:
    print("ERROR: No device connected. Check the USB cable and IviumSoft.")

except NoDeviceDetectedError:
    print("ERROR: Device not detected. Check USB drivers.")

except DeviceBusyError:
    print("ERROR: Device is busy. Wait for the current measurement to finish.")

finally:
    try:
        Pyvium.close_driver()
    except DriverNotOpenError:
        pass  # open_driver() may have failed before we could close

## 5. Checking IviumSoft is Running

Before diving into measurements you can probe the system state without raising exceptions.

In [ ]:
Pyvium.open_driver()

running = Pyvium.is_iviumsoft_running()
print(f"IviumSoft running: {running}")

if running:
    status_code, status_label = Pyvium.get_device_status()
    print(f"Device status: {status_code} → '{status_label}'")
    # status_code meanings:
    #  -1 : no IviumSoft
    #   0 : not connected
    #   1 : available, idle
    #   2 : available, busy (running a method)
    #   3 : no device available

Pyvium.close_driver()

## 6. Minimal Working Example

The simplest complete script: open the driver, confirm IviumSoft is running, then close.

In [ ]:
from pyvium import Pyvium
from pyvium.errors import IviumSoftNotRunningError

try:
    Pyvium.open_driver()

    if not Pyvium.is_iviumsoft_running():
        raise IviumSoftNotRunningError()

    status_code, status_label = Pyvium.get_device_status()
    print(f"Status: {status_label}")
    print(f"DLL OK: {Pyvium.check_dll_version()}")
    print("Setup verified — ready to proceed")

finally:
    try:
        Pyvium.close_driver()
    except Exception:
        pass

---

## Summary

| Concept | Key points |
|---------|------------|
| Import | `from pyvium import Pyvium` |
| Always first | `Pyvium.open_driver()` |
| Always last | `Pyvium.close_driver()` (use `try/finally`) |
| Version check | `Pyvium.check_dll_version()` should return `True` |
| Device state | `Pyvium.get_device_status()` → `(code, label)` |
| Error types | 8 typed exceptions — 6 for device/connection state, 2 new for setter result codes |

## DLL return codes (Core layer)

If you ever use the `Core` layer directly, all DLL functions return an integer status code:

| Code | Meaning |
|------|---------| 
| `0` | Success |
| `-1` | No device / IviumSoft not running |
| `1` | Illegal command |
| `2` | Argument out of range |
| `3` | Invalid state |

The `Pyvium` layer translates these into typed exceptions automatically — you only see the raw codes if you use `Core` directly.

## Next Steps

- **`02_device_and_instance_management.ipynb`** — Select devices, manage multiple IviumSoft instances, connect by serial number
- **`03_direct_mode_basics.ipynb`** — Control potential/current directly without a method file
- **`07_data_processing.ipynb`** — Parse IDF files offline (no hardware required)